# Lab 18 — Train a Churn Model on the GPU

You have already run a **Spark job** that turned 849 MB of raw CSV into 232 MB of Parquet.
This notebook does everything that comes after: it builds the training data, trains a model
on the GPU, and registers it in MLflow under your name.

---

## What you will do

| Part | What happens | Concept |
|---|---|---|
| 1 | Set your student number, check the GPU | environment |
| 2 | Look at what Spark produced | Parquet, partitions |
| 3 | Same job on CPU, then GPU — **timed** | **RAPIDS / cuDF** |
| 4 | Fetch the answer from a different system | the label |
| 5 | Join behaviour to identity on the GPU | building a training table |
| 5b | Run the validation checklist | data readiness |
| 6 | Train a model | XGBoost on GPU |
| 7 | See which columns mattered | feature importance |
| 8 | Register the model under your name | MLflow |
| 9 | Same model, scikit-learn vs cuML — **timed** | **CUDA-X: cuML** |
| 10 | Find look-alike subscribers on the GPU | **CUDA-X: cuVS** |
| 11 | Turn your numbers into a value statement | presales |

## Before you start

- ✅ Your Spark job finished — you should have a `curated/student-NN/` folder
- ✅ You are on a **GPU** notebook
- ✅ You know your **student number**

## How to run this notebook

> **Run the cells in order, from the top.** Use `Shift + Enter`, or
> **Kernel → Restart & Run All**.
>
> ⚠️ Skipping ahead will fail — later cells use variables defined earlier. If you get a
> `NameError`, you skipped a cell. Restart and run from the top.


---
# Part 1 — Setup

## 1.1 — Check the environment, and fix it if needed

**▶ Run this cell first.**

On a properly built image this does nothing and prints `environment ready` — carry
straight on. It only installs when something is actually missing or too old.

**If it does install something, it will tell you to restart the kernel.** Do that, then
start again from the top. The restart is unavoidable: NumPy is compiled into other
packages, so changing it under a running kernel leaves them half-loaded against the old
version — the classic `numpy.dtype size changed` error.

### Why this is fiddly

Some images ship `scikit-learn 1.3` and `scipy 1.11`, which predate NumPy 2 and actively
hold NumPy at 1.x. They have to be *upgraded*, not just installed. Separately, mlflow
wants `pandas<3` while the GPU dataframe library wants `pandas>=3` — so the packages are
installed first and the NumPy/pandas pair is corrected afterwards, rather than asking pip
to reconcile a genuine contradiction. mlflow prints a version warning and works fine.

> **cuDF is never installed here** — 1–2 GB and it needs its own restart. Either your
> image has it or the notebook runs on the CPU, which works fine minus the speed comparison.

In [ ]:
import sys, subprocess, importlib.util

def pip(*a):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=False)

def version_of(mod):
    try:
        import importlib.metadata as md
        return tuple(int(x) for x in md.version(mod).split(".")[:2])
    except Exception:
        return None

HAS_CUDF = importlib.util.find_spec("cudf") is not None
# CUDA-X extras used in Parts 9-10. Report only -- like cuDF they must be baked into
# the image (each is >1 GB and needs a kernel restart). Absent => those parts fall
# back to CPU and you lose the timed comparison, nothing else.
HAS_CUML = importlib.util.find_spec("cuml") is not None
HAS_CUVS = importlib.util.find_spec("cuvs") is not None

# ── is the environment already good? ──────────────────────────────────────
problems = []
np_v, pd_v = version_of("numpy"), version_of("pandas")
sk_v, sp_v = version_of("scikit-learn"), version_of("scipy")

if not np_v or np_v[0] < 2:            problems.append("numpy < 2")
if not sk_v or sk_v < (1, 5):          problems.append("scikit-learn < 1.5 (numpy-1 only)")
if not sp_v or sp_v < (1, 13):         problems.append("scipy < 1.13 (numpy-1 only)")
if HAS_CUDF and (not pd_v or pd_v[0] < 3):  problems.append("pandas < 3 but cuDF needs 3.x")
if not HAS_CUDF and (not pd_v or pd_v < (2, 2)): problems.append("pandas < 2.2")
for m in ("psycopg2", "xgboost", "mlflow"):
    if importlib.util.find_spec(m) is None: problems.append(f"{m} missing")
# PCAI runs an MLflow 2.x SERVER. A 3.x client calls /api/2.0/mlflow/logged-models,
# which that server does not have -> 404, and model registration fails.
mf_v = version_of("mlflow")
if mf_v and mf_v[0] >= 3: problems.append("mlflow 3.x client vs 2.x server (breaks registry)")

if not problems:
    import numpy, pandas, importlib.metadata as md
    print(f"numpy {numpy.__version__} · pandas {pandas.__version__}"
          f"{' · cuDF present' if HAS_CUDF else ' · no cuDF (CPU path)'}"
          f"{' · cuML' if HAS_CUML else ' · no cuML'}{' · cuVS' if HAS_CUVS else ' · no cuVS'}")
    # The check above read versions from package metadata ON DISK. Make sure the
    # kernel actually IMPORTS those versions -- if not, either the kernel was not
    # restarted after an install, or pip installed into a location this kernel
    # does not import from (two site-packages trees, one stale).
    mismatch = [(m, mod.__version__, md.version(m)) for m, mod in
                (("numpy", numpy), ("pandas", pandas)) if mod.__version__ != md.version(m)]
    if mismatch:
        print("\n⚠️  the kernel imports DIFFERENT versions than pip installed:")
        for m, imp, disk in mismatch:
            print(f"     {m}: imported {imp}, on disk {disk}")
        print(f"     numpy lives at  {numpy.__file__}")
        print(f"     pandas lives at {pandas.__file__}")
        print("  -> Kernel -> Restart & Run All. If this message persists after a")
        print("     restart, the image has two package trees and must be rebuilt.")
    else:
        print("\nenvironment ready — no restart needed, carry on to 1.2")
else:
    print("needs fixing:")
    for x in problems: print("  -", x)

    # STEP 1 — upgrade freely. scikit-learn/scipy MUST be upgraded, not just installed:
    # the old versions pin numpy below 2 and will drag it back down.
    print("\ninstalling ...")
    pip("--upgrade", "psycopg2-binary", "xgboost", "mlflow>=2.9,<3",
        "scikit-learn>=1.5", "scipy>=1.13")

    # STEP 2 — now force the numpy/pandas pair. Last install wins.
    pip("numpy>=2,<3", "pandas>=3.0,<3.0.4" if HAS_CUDF else "pandas>=2.2.3,<3")

    print("\n" + "=" * 62)
    print("  RESTART THE KERNEL NOW, then Kernel -> Restart & Run All")
    print("  (numpy is compiled into other packages; a restart is required)")
    print("=" * 62)

## 1.2 — Your student number

**▶ Edit the number below to your own, then run the cell.**

This is the only thing in the notebook you change. Everything else — which folder you read,
what your experiment is called, what your model is named — is built from this number, so
everyone in the room works independently without collisions.

*(Note the absolute path. A notebook's working directory is your home folder, not wherever
you think it is, so we never use relative paths.)*

In [ ]:
import os

STUDENT_ID = 0                       # ←←← CHANGE THIS TO YOUR NUMBER

SID     = f'{STUDENT_ID:02d}'        # zero-padded: 0 -> '00', 7 -> '07'
LAB     = os.path.expanduser('~/shared/data-engineering-lab')
CURATED = f'{LAB}/curated/student-{SID}/watch_events'
CUTOFF  = '2026-05-31'               # we use the LAST 30 DAYS of history as features

print(f'You are student {SID}')
print(f'Reading from : {CURATED}')
print(f'Feature window: events on or after {CUTOFF}')

assert os.path.isdir(CURATED), (
    f'\nNo curated data at {CURATED}\n'
    f'Either your Spark job has not finished, or STUDENT_ID is wrong.')
print('\n✅ Your curated data is there — carry on.')

## 1.3 — GPU check, and a courtesy to your neighbours

**▶ Run this cell. It must run before any other cell that touches the GPU.**

You are sharing a physical GPU with other people. Sharing splits the *compute*, but **not the
memory** — so if one notebook grabs all the GPU memory, everyone else on that card crashes.

The cell below caps this notebook at 1.5 GB of GPU memory. Everything we do fits easily inside
that. If you restart the kernel later, run this cell first again.

**What is cuDF?** It is a dataframe library that looks *exactly* like pandas but runs on the
GPU. Same method names, same syntax — different hardware. It is part of **RAPIDS**, NVIDIA's
suite of GPU data-science libraries. You will see the difference it makes in Part 3.

👀 **Look for:** the GPU name, and `cuDF ... available`. If cuDF is missing the notebook still
works, it just falls back to the CPU and you miss the speed comparison.

In [ ]:
import subprocess

try:
    import rmm                                    # RAPIDS memory manager
    rmm.reinitialize(pool_allocator=True,
                     initial_pool_size='256MB',
                     maximum_pool_size='1500MB')  # ← the cap that protects everyone else
    import cudf
    HAS_GPU_DF = True
    print(f'✅ cuDF {cudf.__version__} available — GPU memory capped at 1.5 GB')
except Exception as e:
    HAS_GPU_DF, cudf = False, None
    print('⚠️  cuDF not available — the notebook will use the CPU instead.')
    print('   (', str(e)[:70], ')')

print()
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.used',
                      '--format=csv'], capture_output=True, text=True).stdout)

## 1.4 — Can this notebook talk to MLflow?

**▶ Run this cell.** It sends one harmless request to the MLflow server with the login
token this notebook was given. Part 8 needs this to work, and it is far better to find out
now than after training.

👀 **Look for:** `MLflow reachable and token accepted`. If it reports a token problem, tell
the instructor and carry on — everything up to Part 8 still works without MLflow.


In [ ]:
import os, urllib.request, urllib.error, base64, json

MLFLOW_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://mlflow.mlflow.svc.cluster.local:5000')
TOKEN_FILE = '/etc/secrets/ezua/.auth_token'

# Prefer the platform-refreshed token file; fall back to whatever the pod was given.
if os.path.exists(TOKEN_FILE) and open(TOKEN_FILE).read().strip():
    os.environ['MLFLOW_TRACKING_TOKEN'] = open(TOKEN_FILE).read().strip()
    token_src = TOKEN_FILE
else:
    token_src = 'MLFLOW_TRACKING_TOKEN env var' if os.environ.get('MLFLOW_TRACKING_TOKEN') else 'none'
tok = os.environ.get('MLFLOW_TRACKING_TOKEN', '')

def _issuer(t):
    try:
        p = t.split('.')[1]; p += '=' * (-len(p) % 4)
        return json.loads(base64.urlsafe_b64decode(p)).get('iss', '?')
    except Exception:
        return 'not a JWT'

req = urllib.request.Request(f'{MLFLOW_URI}/api/2.0/mlflow/experiments/search?max_results=1',
                             headers={'Authorization': f'Bearer {tok}'} if tok else {})
try:
    with urllib.request.urlopen(req, timeout=10) as r:
        print(f'✅ MLflow reachable and token accepted   ({MLFLOW_URI})')
        print(f'   token from: {token_src}')
except urllib.error.HTTPError as e:
    print(f'⚠️  MLflow answered HTTP {e.code} at {MLFLOW_URI}')
    print(f'   token from : {token_src}')
    print(f'   token issuer: {_issuer(tok) if tok else "no token at all"}')
    if e.code in (401, 403):
        print('\n   MLflow rejected the login token. It only accepts a USER token from the')
        print('   platform login (issuer contains "keycloak"). A Kubernetes service-account')
        print('   token, or no token, is refused. The platform normally provides the user')
        print('   token in /etc/secrets/ezua/.auth_token for a notebook in YOUR OWN project.')
        print('   -> Tell the instructor. Parts 2-7 work regardless; only Part 8 needs this.')
except Exception as e:
    print(f'⚠️  MLflow not reachable at {MLFLOW_URI}: {str(e)[:80]}')
    print('   Parts 2-7 work regardless; only Part 8 needs MLflow.')


---
# Part 2 — What did Spark actually produce?

## The format change, and why it matters

Your Spark job did **not** change the data. Same 20,010,929 rows went in and came out. What it
changed was the **format**:

```
CSV      stored ROW by row, as text
         168654,5,478,2026-04-11,4.1,web,1
         168655,8,1902,2026-04-11,54.9,tablet,1

Parquet  stored COLUMN by column, in binary
         [all the event_ids] [all the subscriber_ids] [all the watch_minutes] ...
```

Two consequences, and you will feel both in Part 3:

1. **It is 3.66× smaller** — 849 MB became 232 MB. A column of repeated values
   (`tv, tv, mobile, tv`) compresses beautifully when those values sit together.
2. **You can read one column without reading the others.** We need 4 of the 7 columns.
   Parquet reads just those 4. CSV would have to read every character of every line to find
   them.

This is called **columnar storage**, and it is why analytics systems everywhere use Parquet
rather than CSV.

## Partitions — only read what you need

Spark wrote one folder per day: `dt=2026-01-01`, `dt=2026-01-02`, and so on — 180 of them.

We only want the **last 30 days**, so we read 30 folders and never open the other 150. That is
**partition pruning**: a real speed-up that comes purely from how the files were named.

**▶ Run the cell.** 👀 **Look for:** 180 total, 30 kept.

In [ ]:
# Guard: this needs the Config cell in Part 1. Running cells out of order is the most
# common mistake, and a bare NameError does not explain it.
for _v in ('SID', 'CURATED', 'CUTOFF'):
    if _v not in globals():
        raise RuntimeError(f"'{_v}' is not defined — run Part 1 first "
                           "(Kernel -> Restart & Run All).")

import glob, datetime

cutoff   = datetime.date.fromisoformat(CUTOFF)
day_dirs = sorted(glob.glob(f'{CURATED}/dt=*'))
keep     = [d for d in day_dirs
            if datetime.date.fromisoformat(d.split('dt=')[1]) >= cutoff]
files    = [f for d in keep for f in glob.glob(f'{d}/*.parquet')]

print(f'day-folders Spark wrote   : {len(day_dirs)}')
print(f'day-folders we will read  : {len(keep)}   ← the other '
      f'{len(day_dirs)-len(keep)} are never opened')
print(f'parquet files to read     : {len(files)}')

---
# Part 3 — The RAPIDS moment: CPU vs GPU

## What we are about to compute

The raw data has **one row per viewing session** — roughly 100 rows per person. A model needs
**one row per person**. So for each subscriber we roll up their last 30 days:

| From (many rows) | To (one row) |
|---|---|
| every viewing | `minutes_30d` — total minutes watched |
| | `sessions_30d` — how many times they watched |
| | `completion_rate` — fraction they finished |
| | `distinct_titles_30d` — how many different titles |

In SQL this is `GROUP BY`. In pandas and cuDF it is `.groupby()`. We call it **the squash**.

These four columns are **features** — facts about a person, computed from their behaviour.
They do not exist anywhere in your data. You are inventing them. That invention is most of
applied machine learning.

---

## ✋ Before you run the next two cells

We will do the **identical** calculation twice — once on the CPU with pandas, once on the GPU
with cuDF. The code is the same function both times.

> **Write down your guess: how many times faster will the GPU be?**

**▶ Run the CPU cell first.**

In [ ]:
import pandas as pd, time

COLS = ['subscriber_id', 'watch_minutes', 'completed', 'content_id']

def squash(df):
    """Many rows per person -> one row per person.
    This exact function runs on BOTH pandas and cuDF — same API, different hardware."""
    g = df.groupby('subscriber_id').agg(
        {'watch_minutes': ['sum', 'count'],
         'completed': 'mean',
         'content_id': 'nunique'})
    g.columns = ['minutes_30d', 'sessions_30d', 'completion_rate', 'distinct_titles_30d']
    return g.reset_index()

print('Reading with pandas (CPU) ...')
t0 = time.time()
pdf = pd.concat([pd.read_parquet(f, columns=COLS) for f in files], ignore_index=True)
t_read_cpu = time.time() - t0

print('Squashing with pandas (CPU) ...')
t0 = time.time()
usage_cpu = squash(pdf)
t_agg_cpu = time.time() - t0

print(f'\n  rows read       : {len(pdf):,}')
print(f'  people out      : {len(usage_cpu):,}')
print(f'  CPU read time   : {t_read_cpu:6.2f} s')
print(f'  CPU squash time : {t_agg_cpu:6.2f} s')
usage_cpu.head()

## Now the same thing on the GPU

**▶ Run this cell.** Nothing about the calculation changes — `squash()` is the same function.
Only the library differs: `cudf.read_parquet` instead of `pd.read_parquet`.

👀 **Look for:** the speed-up line at the bottom. Compare it to your guess.

*(The GPU read includes a one-time start-up cost, so the read number is pessimistic. The
squash number is the honest comparison.)*

In [ ]:
if HAS_GPU_DF:
    print('Reading with cuDF (GPU) ...')
    t0 = time.time()
    gdf = cudf.read_parquet(files, columns=COLS)
    t_read_gpu = time.time() - t0

    print('Squashing with cuDF (GPU) ...')
    t0 = time.time()
    usage_gpu = squash(gdf)          # ← the SAME function as the CPU cell
    t_agg_gpu = time.time() - t0

    print(f'\n  rows read       : {len(gdf):,}')
    print(f'  GPU read time   : {t_read_gpu:6.2f} s')
    print(f'  GPU squash time : {t_agg_gpu:6.2f} s')
    print('\n' + '=' * 52)
    print(f'  SQUASH SPEED-UP : {t_agg_cpu / max(t_agg_gpu, 1e-6):5.1f}x'
          f'   (CPU {t_agg_cpu:.2f}s  ->  GPU {t_agg_gpu:.2f}s)')
    print('=' * 52)
    usage = usage_gpu.to_pandas()
else:
    print('cuDF not available — using the CPU result from the previous cell.')
    usage = usage_cpu

print(f'\n{len(usage):,} people, one row each. This is HALF of the training data —')
print('it says what people DID, but not whether they cancelled.')
usage.head()

---
# Part 4 — The answer lives somewhere else

Everything so far has been **behaviour** — what people watched. But the thing we want to
predict, `churned_next_30d`, is **not in the event data at all**.

It lives in a completely different system: a **Postgres database** holding the customer
records — plan, tenure, support tickets, and whether they cancelled.

```
watch_events   →  files on shared storage  →  what people DID       (no answer)
subscribers    →  a Postgres database      →  who people ARE + THE ANSWER
```

**This is the point of the whole exercise.** In any real company, activity logs and customer
records live in different systems owned by different teams. You cannot train anything until
you bring them together.

### Why the answer matters

A model learns by example. You show it thousands of people where you *already know* what
happened, it works out the pattern, and then it can predict for someone new.

> Past exam papers **with the answer key** are your training data. The real exam has no answer
> key — that is prediction. You cannot learn from past papers if nobody wrote the answers down.

`churned_next_30d` is the answer key.

**▶ Run the cell.** 👀 **Look for:** 200,000 rows and a churn rate near 0.12 — about 12% of
people cancelled.

In [ ]:
# psycopg2 is small and shares no dependencies with numpy/pandas/cuDF, so unlike
# cuDF it is safe to install inline — no kernel restart needed. The version pins
# stop pip from quietly downgrading numpy underneath cuDF.
try:
    import psycopg2
except ImportError:
    import sys, subprocess
    print('installing psycopg2-binary ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'psycopg2-binary', 'numpy>=2,<3', 'pandas>=3.0,<3.0.4'],
                   check=False)
    import psycopg2
    print('ok')


PG = dict(host='postgres-data-postgresql.postgres-data.svc.cluster.local',
          port=5432, dbname='app_db',
          user='admin', password='admin',      # read-only account
          connect_timeout=10)

conn = psycopg2.connect(**PG)
subs_pd = pd.read_sql('''
    SELECT subscriber_id, plan, country, payment_method, monthly_price,
           support_tickets_90d, tenure_months, churned_next_30d
    FROM   public.subscribers''', conn)
conn.close()

print(f'subscribers : {len(subs_pd):,} rows')
print(f'churn rate  : {subs_pd.churned_next_30d.mean():.4f}   '
      f'({subs_pd.churned_next_30d.sum():,} people cancelled)')
subs_pd.head()

---
# Part 5 — Join the two halves, on the GPU

200,000 rows of behaviour meet 200,000 rows of identity. `subscriber_id` is the link.

## One subtlety worth understanding

We use a **LEFT join** starting from `subscribers`, not an inner join.

Why: somebody who watched **nothing** in the last 30 days has no row in the behaviour table.
An inner join would quietly drop them — and those are exactly the people most likely to have
cancelled. So we keep every subscriber and fill their missing activity with **zeros**.

> **Absence of a record is itself information.** Dropping those rows would delete the
> strongest signal in the dataset, and the code would look perfectly correct.

**▶ Run the cell.** 👀 **Look for:** the two-row table at the bottom. People who churned watch
far fewer minutes than people who stayed. **That gap is what the model will learn.**

In [ ]:
def to_gpu(pdf):
    """pandas -> cuDF. The function name moved between cuDF versions, so try both."""
    return cudf.from_pandas(pdf) if hasattr(cudf, 'from_pandas') else cudf.DataFrame(pdf)

FILL = {'minutes_30d': 0.0, 'sessions_30d': 0,
        'completion_rate': 0.0, 'distinct_titles_30d': 0}

if HAS_GPU_DF:
    feat_df = (to_gpu(subs_pd)
               .merge(to_gpu(usage), on='subscriber_id', how='left')
               .fillna(FILL)
               .to_pandas())
    where = 'GPU'
else:
    feat_df = (subs_pd.merge(usage, on='subscriber_id', how='left').fillna(FILL))
    where = 'CPU'

print(f'Training table ({where} join): {len(feat_df):,} rows x {feat_df.shape[1]} columns\n')

summary = (feat_df.groupby('churned_next_30d')
                  .agg(people=('subscriber_id', 'count'),
                       avg_minutes_30d=('minutes_30d', 'mean'),
                       avg_sessions=('sessions_30d', 'mean')).round(1))
summary.index = ['stayed (0)', 'churned (1)']
print(summary)
print('\n👉 People who cancelled watched far less. That gap IS the model.')

## 5b — The validation checklist, executed

A transformation without a validation is a rumour. Before training on this table, prove it
is what we think it is. Each line below is one row of
`TRANSFORMATION_VALIDATION_CHECKLIST.md`; the number in brackets is the row it checks.

**▶ Run the cell.** 👀 **Look for:** `ALL PASS`. If anything fails, do not continue — go back
to the part that produced that column.


In [ ]:
checks = [
 ('[3.2] LEFT join kept every subscriber',   len(feat_df) == len(subs_pd),
      f'{len(feat_df):,} in, {len(subs_pd):,} out'),
 ('[3.4] join key unique on both sides',     subs_pd.subscriber_id.is_unique and usage.subscriber_id.is_unique,
      'subscriber_id unique'),
 ('[3.3] no nulls after fill',               int(feat_df.isna().sum().sum()) == 0,
      f'{int(feat_df.isna().sum().sum())} nulls'),
 ('[3.5] zero-activity group preserved',     int((feat_df.sessions_30d == 0).sum()) > 0,
      f'{int((feat_df.sessions_30d == 0).sum()):,} people watched nothing; '
      f'{feat_df.loc[feat_df.sessions_30d == 0, "churned_next_30d"].mean():.0%} of them churned'),
 ('[2.5] completion_rate within [0, 1]',     bool(feat_df.completion_rate.between(0, 1).all()),
      f'max {feat_df.completion_rate.max():.3f}'),
 ('[2.5] counts non-negative',               bool((feat_df[['minutes_30d','sessions_30d','distinct_titles_30d']] >= 0).all().all()),
      'ok'),
 ('[0.5] label base rate near 0.12',         0.10 < feat_df.churned_next_30d.mean() < 0.14,
      f'{feat_df.churned_next_30d.mean():.4f}'),
 ('[3.6] churners watch less than stayers',  bool(feat_df.groupby('churned_next_30d').minutes_30d.mean().diff().iloc[-1] < 0),
      'mean minutes ' + str(feat_df.groupby('churned_next_30d').minutes_30d.mean().round(0).astype(int).to_dict())),
]
if HAS_GPU_DF:
    same = (len(usage_cpu) == len(usage_gpu)
            and abs(usage_cpu.minutes_30d.sum() - float(usage_gpu.minutes_30d.sum())) < 1e-3)
    checks.append(('[2.4] CPU and GPU squash identical', bool(same),
                   f'{len(usage_cpu):,} rows, same total minutes'))

width = max(len(c[0]) for c in checks)
for name, ok, detail in checks:
    print(('✅ PASS  ' if ok else '❌ FAIL  ') + name.ljust(width) + '   ' + detail)
n_fail = sum(not c[1] for c in checks)
print('\n' + ('ALL PASS — this table is ready to train on.' if n_fail == 0
               else f'{n_fail} FAILED — fix before training.'))
assert n_fail == 0, 'validation failed; see above'


---
# Part 6 — Train the model

We use **XGBoost**, a gradient-boosted tree model — the standard choice for tabular data like
this.

Two things happen first:

1. **One-hot encoding.** A model needs numbers, not words. `plan` becomes three 0/1 columns:
   `plan_basic`, `plan_standard`, `plan_premium`.
2. **Train/test split.** We train on 75% and keep 25% hidden. Scoring on data the model has
   already seen would tell us nothing — it could simply memorise.

`device='cuda'` puts the training on the GPU. One keyword; no rewrite.

## Reading the score

**AUC** measures how well the model separates the two groups. Pick a churner and a non-churner
at random: AUC is the probability the model scores the churner higher.

| AUC | Meaning |
|---|---|
| 0.5 | Useless — coin flip |
| 0.7 | Some signal |
| **~0.87** | **Good — what you should get** |
| 1.0 | Perfect. Be suspicious: usually a leak |

**▶ Run the cell.** 👀 **Look for:** AUC around 0.87.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

y = feat_df['churned_next_30d']
X = feat_df.drop(columns=['subscriber_id', 'churned_next_30d'])
X = pd.get_dummies(X, columns=['plan', 'country', 'payment_method'])  # words -> numbers

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)

print(f'training on {len(X_tr):,} people, testing on {len(X_te):,} held-out people')
print(f'{X.shape[1]} features after one-hot encoding\n')

def make(device):
    return xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                             subsample=0.9, tree_method='hist', device=device,
                             eval_metric='auc')

try:
    model = make('cuda'); model.fit(X_tr, y_tr); dev = 'GPU'
except Exception as e:
    print('GPU training unavailable, falling back to CPU:', str(e)[:70])
    model = make('cpu');  model.fit(X_tr, y_tr); dev = 'CPU'

model.set_params(device='cpu')      # inference on host data; avoids a device-mismatch warning
auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

print(f'\n{"=" * 46}')
print(f'  TEST AUC: {auc:.4f}     ({dev} training)')
print('=' * 46)
print('\n📝 Write this number down — it goes on the leaderboard.')

---
# Part 7 — Which columns actually mattered?

Now the interesting part. Two things to look for, and the second is the real lesson.

### 1. Similar columns share the credit
`minutes_30d`, `sessions_30d` and `distinct_titles_30d` all measure the same underlying thing —
how much someone watches. A tree model picks one or two and largely ignores the rest, so do not
be surprised if `minutes_30d` ranks low. That does **not** mean watch-time is irrelevant; you
saw in Part 5 that churners watch a third as much. Importance shows what the model *used*, not
what the world *contains*.

### 2. Useless columns do not score zero — they score *identically*
`country` was put into this dataset as **deliberate noise**. It predicts nothing. But every
column picks up a little importance by chance, so it will not be zero.

The giveaway is the **spread**. Watch the six `country_*` values land within a hair of each
other while the real features differ by an order of magnitude.

> **Real features disagree with each other. Noise features are indistinguishable.**
> That tight little cluster is the fingerprint of a column carrying nothing — and it is how you
> spot a useless column in practice.

**▶ Run the cell.**

In [ ]:
imp = (pd.Series(model.feature_importances_, index=X.columns)
         .sort_values(ascending=False))

print('TOP 8 FEATURES')
print(imp.head(8).to_string())

noise = imp[[c for c in imp.index if c.startswith('country_')]]
real  = imp[[c for c in imp.index if not c.startswith('country_')]]

print('\nTHE PLANTED NOISE (country)')
print(noise.to_string())

print('\n' + '=' * 58)
print(f'  noise columns : spread {noise.max()-noise.min():.4f}  '
      f'(from {noise.min():.4f} to {noise.max():.4f})')
print(f'  real  columns : spread {real.max()-real.min():.4f}  '
      f'(from {real.min():.4f} to {real.max():.4f})')
print('=' * 58)
print('Real features disagree with each other. Noise features do not.')

---
# Part 8 — Register the model in MLflow

A trained model sitting in a notebook is worthless to everyone else. **MLflow** is the record:
it stores what was trained, with which settings, by whom, and what score it got.

That record is what makes a model:

- **findable** — a colleague can locate your model without asking you
- **comparable** — v1 against v2, side by side
- **auditable** — every parameter and every promotion is logged
- **deployable** — serving reads from the registry, not from your laptop

Your model is registered as **`student-NN-churn`**, so it is yours and appears on the
leaderboard.

**▶ Run the cell.** 👀 **Look for:** the run URL. Open it and look at your parameters, your
metric, and the registered model.

In [ ]:
import mlflow

try:
    uri = os.environ.get('MLFLOW_TRACKING_URI',
                         'http://mlflow.mlflow.svc.cluster.local:5000')
    tok = '/etc/secrets/ezua/.auth_token'
    if os.path.exists(tok):
        os.environ['MLFLOW_TRACKING_TOKEN'] = open(tok).read().strip()
    mlflow.set_tracking_uri(uri)
    mlflow.set_experiment(f'student-{SID}-churn')

    with mlflow.start_run(run_name=f'student-{SID}') as run:
        mlflow.log_params({'n_estimators': 300, 'max_depth': 6,
                           'learning_rate': 0.1, 'subsample': 0.9})
        mlflow.log_metric('auc', auc)
        mlflow.set_tags({'student_id': SID, 'dataset': 'streaming-churn',
                         'rows_trained_on': len(X_tr)})
        mlflow.xgboost.log_model(model, 'model',
                                 registered_model_name=f'student-{SID}-churn')
    print(f'\n✅ Registered as: student-{SID}-churn')
    print(f'   AUC {auc:.4f}')
except Exception as e:
    msg = str(e)
    print('MLflow logging skipped:', msg[:200])
    if '401' in msg or 'Jwt' in msg:
        print('\n→ MLflow rejected the login token. The platform normally puts a USER token')
        print('  in /etc/secrets/ezua/.auth_token; when that file is missing the notebook only')
        print('  has a Kubernetes service-account token, which MLflow does not accept.')
        print('  Ask the instructor: this notebook server needs the ezua auth token mounted.')
    if 'logged-models' in msg or '404' in msg:
        print('\n→ This is an mlflow CLIENT/SERVER version mismatch: a 3.x client calling')
        print('  an endpoint the 2.x server does not have. Fix with:')
        print('      pip install -q "mlflow>=2.9,<3"   then restart the kernel')
    print('\nYour model and AUC above are still valid — only the registry step failed.')

---
# Part 9 — CUDA-X: the same model on cuML

XGBoost already ran on the GPU in Part 6 with one keyword. **cuML** goes further: it is a
GPU implementation of the scikit-learn API. Same class names, same `.fit()` / `.predict_proba()`,
different hardware. NVIDIA's headline is *"50× faster scikit-learn"*; you are about to measure
what it is **on this data, on this GPU, against this CPU**. That measured number is the one
you may quote. The headline is a ceiling.

We train a RandomForest twice with identical parameters — scikit-learn on the CPU cores this
notebook has, cuML on the GPU — and check that the two models agree (AUC within noise) before
looking at the clock. **A fast wrong answer is worth nothing**, so accuracy is checked first.

> ✋ **Write down your guess** for the speed-up before running.

**▶ Run the cell.** 👀 **Look for:** two AUCs that match, then the speed-up line.


In [ ]:
import os, time
import numpy as np
from sklearn.ensemble import RandomForestClassifier

def cpu_cores_allowed():
    """CPU cores this NOTEBOOK may use. os.cpu_count() reports the whole node (e.g. 128),
    but a Kubernetes pod is capped by its cgroup quota -- that cap is the honest number
    to compare a GPU against, and the right thread count for scikit-learn."""
    for path in ('/sys/fs/cgroup/cpu.max',):                      # cgroup v2
        try:
            quota, period = open(path).read().split()
            if quota != 'max': return max(1, int(int(quota) / int(period)))
        except Exception: pass
    try:                                                            # cgroup v1
        q = int(open('/sys/fs/cgroup/cpu/cpu.cfs_quota_us').read())
        p = int(open('/sys/fs/cgroup/cpu/cpu.cfs_period_us').read())
        if q > 0: return max(1, int(q / p))
    except Exception: pass
    return os.cpu_count()

X_tr32, X_te32 = X_tr.astype('float32'), X_te.astype('float32')
y_tr32 = y_tr.astype('int32')
RF = dict(n_estimators=200, max_depth=14, random_state=0)
n_cores = cpu_cores_allowed()

t0 = time.time()
sk_rf = RandomForestClassifier(n_jobs=n_cores, **RF).fit(X_tr32, y_tr32)
t_rf_cpu = time.time() - t0
auc_rf_cpu = roc_auc_score(y_te, sk_rf.predict_proba(X_te32)[:, 1])
print(f'scikit-learn RandomForest : {t_rf_cpu:6.2f} s   AUC {auc_rf_cpu:.4f}   ({n_cores} CPU cores)')

try:
    from cuml.ensemble import RandomForestClassifier as cuRandomForest
    # one tiny warm-up fit so the timing below is the model, not GPU start-up
    cuRandomForest(n_estimators=2, max_depth=2).fit(X_tr32.values[:1000], y_tr32.values[:1000])
    t0 = time.time()
    cu_rf = cuRandomForest(**RF).fit(X_tr32.values, y_tr32.values)
    t_rf_gpu = time.time() - t0
    auc_rf_gpu = roc_auc_score(y_te, np.asarray(cu_rf.predict_proba(X_te32.values))[:, 1])
    print(f'cuML RandomForest         : {t_rf_gpu:6.2f} s   AUC {auc_rf_gpu:.4f}   (GPU)')
    print('\n' + '=' * 60)
    print(f'  AUC gap  : {abs(auc_rf_cpu - auc_rf_gpu):.4f}   (same model within noise)')
    print(f'  SPEED-UP : {t_rf_cpu / max(t_rf_gpu, 1e-6):.1f}x   ({n_cores} CPU cores  ->  1 GPU)')
    print('=' * 60)
    HAS_CUML_RESULT = True
except Exception as e:
    print('cuML not available — CPU result only. (', str(e)[:70], ')')
    t_rf_gpu, auc_rf_gpu, HAS_CUML_RESULT = None, None, False


---
# Part 10 — CUDA-X: look-alike search with cuVS

A different question a retention team actually asks: *"who looks like the people who left?"*

That is a **nearest-neighbour search**. Every subscriber is a point in 19-dimensional space
(the same columns the model used, standardised). For each held-out subscriber we find the 10
most similar subscribers in the training set and ask: *how many of them churned?* That share is
a **model-free churn score** — no training, just similarity. **cuVS** is CUDA-X's vector-search
library; the same operation is what powers retrieval for LLMs, only there the vectors are text
embeddings with hundreds of dimensions.

We run it twice: scikit-learn brute force on the CPU, cuVS brute force on the GPU. Then we check
that the GPU found the **same neighbours** (recall@10) before we look at the clock.

*Why brute force and not an approximate index?* At 150,000 vectors × 19 dimensions, exact search
is cheap and correct. cuVS's approximate indexes (CAGRA, IVF) exist for millions of vectors with
hundreds of dimensions — the embedding case. Using them here would be slower **and** less
accurate; we measured it. The right tool depends on the shape of the data.

**▶ Run the cell.** 👀 **Look for:** recall near 1.0, the speed-up, and the look-alike AUC —
compare it to the trained model's 0.87.


In [ ]:
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

K = 10
scaler = StandardScaler().fit(X_tr32)
V_tr = np.ascontiguousarray(scaler.transform(X_tr32), dtype='float32')   # 150k vectors
V_te = np.ascontiguousarray(scaler.transform(X_te32), dtype='float32')   #  50k queries
y_tr_arr = y_tr.values

t0 = time.time()
nn = NearestNeighbors(n_neighbors=K, algorithm='brute', n_jobs=n_cores).fit(V_tr)
_, nbr_cpu = nn.kneighbors(V_te)
t_knn_cpu = time.time() - t0
score_cpu = y_tr_arr[nbr_cpu].mean(axis=1)            # share of churned neighbours
auc_knn_cpu = roc_auc_score(y_te, score_cpu)
print(f'scikit-learn brute force : {t_knn_cpu:6.2f} s   '
      f'{len(V_te):,} queries x {len(V_tr):,} vectors x {V_tr.shape[1]} dims   ({n_cores} CPU cores)')

try:
    import cupy as cp
    from cuvs.neighbors import brute_force
    # NOTE: a brute-force index REFERENCES the GPU array it was built on; it does not
    # copy it. Keep that array in a variable until the search is done, or the search
    # reads freed memory and returns garbage (recall drops to ~0 -- we hit this).
    warm_v, warm_q = cp.asarray(V_tr[:1000]), cp.asarray(V_te[:10])
    brute_force.search(brute_force.build(warm_v), warm_q, K)            # warm-up
    t0 = time.time()
    V_tr_gpu, V_te_gpu = cp.asarray(V_tr), cp.asarray(V_te)
    index = brute_force.build(V_tr_gpu, metric='sqeuclidean')
    _, nbr_gpu = brute_force.search(index, V_te_gpu, K)
    nbr_gpu = cp.asarray(nbr_gpu).get()
    t_knn_gpu = time.time() - t0
    recall = np.mean([len(set(a) & set(b)) / K for a, b in zip(nbr_cpu, nbr_gpu)])
    score_gpu = y_tr_arr[nbr_gpu].mean(axis=1)
    auc_knn_gpu = roc_auc_score(y_te, score_gpu)
    print(f'cuVS brute force         : {t_knn_gpu:6.2f} s   (GPU)')
    print('\n' + '=' * 60)
    print(f'  recall@{K} vs CPU : {recall:.4f}   (same neighbours)')
    print(f'  SPEED-UP        : {t_knn_cpu / max(t_knn_gpu, 1e-6):.1f}x')
    print('=' * 60)
    HAS_CUVS_RESULT = True
except Exception as e:
    print('cuVS not available — CPU result only. (', str(e)[:70], ')')
    t_knn_gpu, recall, HAS_CUVS_RESULT = None, None, False

print(f'\nlook-alike churn score AUC : {auc_knn_cpu:.4f}   (no model was trained for this)')
print(f'trained XGBoost AUC        : {auc:.4f}')
print('\n👉 Similarity alone gets most of the way. The model earns the rest — and the model')
print('   is what you register, version and serve. Similarity is what you use in a meeting.')


---
# Part 11 — From your numbers to a value statement

You now have measurements nobody handed you. This cell collects them into one table: the raw
material for the last deliverable, in `PRESALES_VALUE_STATEMENT.md`.

The rule for every line you write: **technical result → what it means → why the customer cares.**
Say only the third line to a customer; have the first line ready when they ask *"compared to
what?"*. Never quote a number you did not measure on the data in front of you.

**▶ Run the cell**, then write **three** value statements — at least one about a
data-engineering result (storage, federation, validation), not acceleration.


In [ ]:
def _fmt(x, unit=''):
    return '—' if x is None else f'{x:.2f}{unit}'
def _spd(a, b):
    return '—' if (a is None or b is None) else f'{a / max(b, 1e-6):.1f}x'

rows = [
 ('Columnar storage (Spark → Parquet)',  '849 MB → 232 MB CSV→Parquet (from your Spark job)',        '3.66x smaller'),
 ('Partition pruning',                    f'{len(day_dirs)} day folders written, {len(keep)} read', f'{1 - len(keep)/len(day_dirs):.0%} of files never opened'),
 ('Absence as signal (LEFT join)',        f'{int((feat_df.sessions_30d == 0).sum()):,} zero-activity subscribers kept',
                                          f'{feat_df.loc[feat_df.sessions_30d == 0, "churned_next_30d"].mean():.0%} of them churned vs {feat_df.churned_next_30d.mean():.0%} overall'),
 ('cuDF squash (20M → 200k)',             f'CPU {_fmt(t_agg_cpu, " s")}  GPU {_fmt(t_agg_gpu if HAS_GPU_DF else None, " s")}', _spd(t_agg_cpu, t_agg_gpu if HAS_GPU_DF else None)),
 ('cuML RandomForest',                    f'CPU {_fmt(t_rf_cpu, " s")}  GPU {_fmt(t_rf_gpu, " s")}  ({n_cores} cores)', _spd(t_rf_cpu, t_rf_gpu)),
 ('cuVS look-alike search',               f'CPU {_fmt(t_knn_cpu, " s")}  GPU {_fmt(t_knn_gpu, " s")}', _spd(t_knn_cpu, t_knn_gpu)),
 ('Model quality',                        f'XGBoost AUC {auc:.4f}; look-alike AUC {auc_knn_cpu:.4f}', 'held-out, 50,000 people'),
 ('Reproducibility',                      f'student-{SID}-churn registered in MLflow', 'params + metric + model, versioned'),
]
w0 = max(len(r[0]) for r in rows); w1 = max(len(r[1]) for r in rows)
print('YOUR MEASURED RESULTS  (student ' + SID + ')\n')
for a, b, c in rows:
    print(f'  {a.ljust(w0)}  {b.ljust(w1)}  {c}')
print('\nNow write three value statements: result → meaning → why the customer cares.')
print('See PRESALES_VALUE_STATEMENT.md for worked examples and the things you must not say.')


---
# Optional — beat your own score

You now have a baseline. Try to improve it.

Go back to **Part 6**, change one or two numbers, and re-run Parts 6 → 8:

```python
max_depth      = 8      # deeper trees (was 6)
n_estimators   = 500    # more trees (was 300)
learning_rate  = 0.05   # smaller steps (was 0.1)
```

Re-running Part 8 creates **version 2** of your model in MLflow. Open the registry and compare
v1 and v2 side by side — that is what model versioning is *for*.

⚠️ Do not expect miracles. With one dominant signal, hyperparameters move AUC by a fraction of
a percent. Discovering that is itself worth knowing: **better features beat better
hyperparameters, almost always.**

---
# What you just did

```
  20,010,929 event rows        (what people watched)
          │  cuDF on the GPU — the squash
          ▼
     200,000 people
          │  + the answer, from a Postgres database
          ▼
     200,000 x 12 training table
          │  XGBoost on the GPU
          ▼
     a model, AUC ~0.87, registered under your name
```

## The ideas, not the tools

| Idea | Where you saw it |
|---|---|
| **Columnar storage** | Parquet — 3.66× smaller, reads only the columns you ask for |
| **Partition pruning** | reading 30 day-folders instead of 180 |
| **Aggregation / feature engineering** | the squash — you invented `minutes_30d` |
| **Hardware acceleration** | cuDF — same API, different hardware |
| **Joining across systems** | files + a live database in one training table |
| **Labels** | you can only learn from history where the answer was recorded |
| **Absence as information** | the LEFT join, and filling with zeros |
| **Model registry** | MLflow — findable, comparable, auditable, deployable |
| **Validation before trust** | Part 5b — every transformation proven before training |
| **CUDA-X, measured not quoted** | cuML and cuVS — same API, accuracy checked first, then the clock |
| **Result → meaning → value** | Part 11 — the sentence a customer acts on traces back to a number you measured |

---

**The hard part of machine learning was getting the table right.** Everything after that was
about ten lines of code.